# Model Comparison: Custom CNN vs Transfer Learning

This notebook compares the performance of the Custom CNN baseline model with the Transfer Learning model.

## Overview

- **Overall Metrics**: Accuracy, Precision, Recall, F1-Score (weighted and macro)
- **Per-Class Metrics**: Detailed comparison for each class
- **Visualizations**: Bar charts and comparison plots
- **Summary Report**: Key findings and improvements


In [3]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from typing import Dict, Optional
from datetime import datetime

# Import ModelComparator and ModelSerializer
from compare_models import ModelComparator
from model_serialization import ModelSerializer

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")


Libraries imported successfully!


## 1. Load Metrics

Load metrics from both models. You can either:
- Use default metrics (update values below)
- Load from saved model packages
- Load from JSON files


In [4]:
# Load metrics from saved model files (following example 2 from run_comparison.py)
serializer = ModelSerializer()
models_dir = Path('models/saved_models')

def find_latest_model(version='1.0'):
    """Find the latest saved model with given version"""
    if not models_dir.exists():
        raise FileNotFoundError(f"Models directory not found: {models_dir}")
    
    # Find all model directories matching the version (format: covid19_model_vX.X_YYYYMMDD_HHMMSS)
    version_str = f'v{version}'
    matching_models = []
    for model_dir in models_dir.iterdir():
        if model_dir.is_dir() and version_str in model_dir.name:
            # Extract timestamp from directory name
            try:
                parts = model_dir.name.split('_')
                if len(parts) >= 3:
                    date_str = parts[-2]
                    time_str = parts[-1]
                    timestamp = datetime.strptime(f"{date_str}_{time_str}", "%Y%m%d_%H%M%S")
                    matching_models.append((timestamp, model_dir))
            except:
                continue
    
    if not matching_models:
        raise FileNotFoundError(f"No model found with version {version_str} in {models_dir}")
    
    # Sort by timestamp (newest first) and return the latest
    matching_models.sort(key=lambda x: x[0], reverse=True)
    return matching_models[0][1]

# Load Custom CNN model (v1.0)
print("Loading Custom CNN model...")
try:
    custom_cnn_model_path = find_latest_model('1.0')
    print(f"  Found: {custom_cnn_model_path.name}")
    custom_cnn_package = serializer.load_model(custom_cnn_model_path)
    custom_cnn_metrics = custom_cnn_package['validation_metrics']
    print("  Metrics loaded successfully")
except Exception as e:
    raise RuntimeError(f"Failed to load Custom CNN model: {e}\n"
                      f"Please ensure you have saved a Custom CNN model (v1.0) using ModelSerializer.")

# Load Transfer Learning model (v2.0)
print("\nLoading Transfer Learning model...")
try:
    transfer_learning_model_path = find_latest_model('2.0')
    print(f"  Found: {transfer_learning_model_path.name}")
    tl_package = serializer.load_model(transfer_learning_model_path)
    transfer_learning_metrics = tl_package['validation_metrics']
    print("  Metrics loaded successfully")
except Exception as e:
    raise RuntimeError(f"Failed to load Transfer Learning model: {e}\n"
                      f"Please ensure you have saved a Transfer Learning model (v2.0) using ModelSerializer.")

# Extract class labels (prefer transfer learning if available)
cnn_class_labels = custom_cnn_package['config']['class_labels']
tl_class_labels = tl_package['config']['class_labels']
class_labels = tl_class_labels if tl_class_labels else cnn_class_labels

if not class_labels:
    raise ValueError("No class labels found in either model. Cannot proceed with comparison.")

print("\n" + "=" * 60)
print("METRICS LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Custom CNN Accuracy: {custom_cnn_metrics.get('accuracy', 'N/A'):.4f if isinstance(custom_cnn_metrics.get('accuracy'), (int, float)) else 'N/A'}")
print(f"Transfer Learning Accuracy: {transfer_learning_metrics.get('accuracy', 'N/A'):.4f if isinstance(transfer_learning_metrics.get('accuracy'), (int, float)) else 'N/A'}")
print(f"Number of classes: {len(class_labels)}")
print("=" * 60)


Loading Custom CNN model...
  Found: covid19_model_v1.0_20251215_174929
Loading model from: models/saved_models/covid19_model_v1.0_20251215_174929/covid19_model_v1.0_20251215_174929.joblib
 Model loaded successfully!
   Version: 1.0
   Created: 2025-12-15T17:49:29.542387
   Input shape: [256, 256, 3]
  Metrics loaded successfully

Loading Transfer Learning model...


/root/Covid19-Project/venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 254 variables whereas the saved optimizer has 258 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


RuntimeError: Failed to load Transfer Learning model: No model found with version v2.0 in models/saved_models
Please ensure you have saved a Transfer Learning model (v2.0) using ModelSerializer.

### Manual Override (Optional)

If you want to manually specify model paths instead of using the latest models, uncomment and modify the code below:


In [ ]:
# Uncomment to manually specify model paths instead of using latest models
# custom_cnn_model_path = Path('models/saved_models/covid19_model_v1.0_20251215_111445')
# transfer_learning_model_path = Path('models/saved_models/covid19_model_v2.0_XXXXXX_XXXXXX')
# 
# # Then load them:
# serializer = ModelSerializer()
# custom_cnn_package = serializer.load_model(custom_cnn_model_path)
# custom_cnn_metrics = custom_cnn_package['validation_metrics']
# 
# tl_package = serializer.load_model(transfer_learning_model_path)
# transfer_learning_metrics = tl_package['validation_metrics']
# 
# class_labels = tl_package['config']['class_labels'] if 'class_labels' in tl_package['config'] else custom_cnn_package['config']['class_labels']

print("Metrics are automatically loaded from latest saved models.")
print("Uncomment the code above to manually specify model paths if needed.")


## 2. Run Model Comparison

Create ModelComparator and run full comparison analysis:


In [ ]:
# Create ModelComparator instance
output_dir = Path('comparison_results')
comparator = ModelComparator(
    custom_cnn_metrics=custom_cnn_metrics,
    transfer_learning_metrics=transfer_learning_metrics,
    class_labels=class_labels,
    output_dir=output_dir
)

# Run full comparison (includes overall metrics, per-class metrics, visualizations, and summary)
comparator.run_full_comparison()


In [ ]:
# The comparison has already been run above, which includes all visualizations.
# If you want to access the comparison DataFrames for further analysis, you can run:

# Get comparison DataFrames
comparison_df = comparator.compare_overall_metrics()
per_class_df = comparator.compare_per_class_metrics()

# You can now use these DataFrames for additional analysis if needed
print("\nComparison DataFrames are available:")
print(f"  - comparison_df: Overall metrics comparison")
print(f"  - per_class_df: Per-class metrics comparison")


## 3. Additional Analysis (Optional)

If you need to perform additional analysis on the comparison results, you can access the DataFrames:


In [ ]:
# Example: Display comparison DataFrames
print("=" * 60)
print("OVERALL METRICS COMPARISON")
print("=" * 60)
print(comparison_df.to_string(index=False))

print("\n" + "=" * 60)
print("PER-CLASS METRICS COMPARISON")
print("=" * 60)
print(per_class_df.to_string(index=False))


In [ ]:
# Visualizations have already been generated by run_full_comparison()
# If you want to regenerate them separately, you can use:
# comparator.visualize_overall_metrics(comparison_df)
# comparator.visualize_per_class_metrics(per_class_df)

print("All visualizations have been generated and saved to the output directory.")


## 4. Summary Report

The summary report has already been generated by `run_full_comparison()`. 
All results have been saved to the output directory.


In [ ]:
# The summary report has already been generated by run_full_comparison()
# You can regenerate it if needed:
comparator.generate_summary_report(comparison_df, per_class_df)


## 5. Results Saved

All comparison results have been automatically saved by `ModelComparator`:
- Overall metrics comparison (CSV and PNG)
- Per-class metrics comparison (CSV and PNG)
- Summary report (TXT)

Files are saved in the `comparison_results` directory.


In [ ]:
# List saved files
print("=" * 60)
print("SAVED FILES")
print("=" * 60)
for file in sorted(output_dir.glob('*')):
    if file.is_file():
        size_kb = file.stat().st_size / 1024
        print(f"  {file.name} ({size_kb:.2f} KB)")
print("=" * 60)
